# Hand Gesture Recognition — ML Experiments (v2 — Final)

## Experiments covered
- **Experiment 1 — Hand Activity Identification** (target: `gesture_label`)
- **Experiment 2 — Person Identification** (target: `person_id`)

## Gesture label mapping
| Label | Gesture |
|-------|---------|
| 0 | up |
| 1 | down |
| 2 | play |
| 3 | pause |
| 4 | stop |

**Note:** Data is already scaled (pre-processing done in Try_4.0.ipynb).

In [ ]:
# ============================================================
# CELL 1 — IMPORTS
# ============================================================
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score)
from sklearn.naive_bayes       import GaussianNB
from sklearn.tree              import DecisionTreeClassifier
from sklearn.ensemble          import RandomForestClassifier
from sklearn.neighbors         import KNeighborsClassifier
from sklearn.linear_model      import LogisticRegression
from sklearn.svm               import SVC
import xgboost as xgb

print('All libraries imported successfully.')

In [ ]:
# ============================================================
# CELL 2 — LOAD DATA
# Replace the path below with your actual full dataset path.
# ============================================================
df = pd.read_excel('SMALL_sample_for_test_Hand_gesture_check.xlsx')  # ← change path for full data

print(f'Dataset shape : {df.shape}')
print(f'Columns       : {df.columns.tolist()}')
print(f'\nGesture labels : {sorted(df["gesture_label"].unique())}')
print(f'Person IDs     : {sorted(df["person_id"].unique())}')
print(f'\nGesture distribution:\n{df["gesture_label"].value_counts().sort_index()}')
print(f'\nPerson  distribution:\n{df["person_id"].value_counts().sort_index()}')

In [ ]:
# ============================================================
# CELL 3 — GLOBAL CONFIGURATION
# ============================================================

# Train-test split ratios  (label → test fraction)
SPLIT_RATIOS = {
    '80-20': 0.20,
    '70-30': 0.30,
    '60-40': 0.40,
    '50-50': 0.50
}

# Percentage of total data to use in each run
DATA_PERCENTAGES = [100, 75, 50, 25]

# Number of folds for Stratified K-Fold
N_FOLDS = 5

print('Configuration set.')

In [ ]:
# ============================================================
# CELL 4 — UTILITY FUNCTIONS
# ============================================================

def get_models():
    """
    Returns a fresh dict of all 7 classifiers.
    Called inside loops so each run gets a clean (untrained) model.

    XGBoost notes:
      • eval_metric='mlogloss'  → suppresses the default metric warning
      • verbosity=0             → silences all XGBoost output
      • Labels MUST be 0-indexed integers → use remap_labels_for_xgboost()
    """
    return {
        'Naive Bayes'        : GaussianNB(),
        'Decision Tree'      : DecisionTreeClassifier(random_state=42),
        'XGBoost'            : xgb.XGBClassifier(
                                    eval_metric='mlogloss',
                                    verbosity=0,
                                    random_state=42),
        'Random Forest'      : RandomForestClassifier(n_estimators=100, random_state=42),
        'KNN'                : KNeighborsClassifier(n_neighbors=5),
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
        'SVM'                : SVC(kernel='rbf', random_state=42)
    }


def remap_labels_for_xgboost(y):
    """
    XGBoost requires class labels to be 0, 1, 2, …
    This is critical when person_id values are like 27, 50, 103 —
    XGBoost would allocate a huge output layer if labels are not remapped.

    Returns:
        y_remapped : array with labels 0…k-1
        label_map  : dict {original_label → new_label}
                     Use this SAME map for test labels to avoid mismatch.
    """
    unique_labels = sorted(np.unique(y))
    label_map = {orig: new for new, orig in enumerate(unique_labels)}
    y_remapped = np.array([label_map[val] for val in y])
    return y_remapped, label_map


def get_metrics(y_true, y_pred):
    """
    Returns (accuracy, precision, recall, f1) all in %.
    Uses 'weighted' average for multi-class consistency.
    zero_division=0 prevents errors when a class has no predictions.
    """
    acc  = round(accuracy_score (y_true, y_pred)                                  * 100, 2)
    prec = round(precision_score(y_true, y_pred, average='weighted', zero_division=0) * 100, 2)
    rec  = round(recall_score   (y_true, y_pred, average='weighted', zero_division=0) * 100, 2)
    f1   = round(f1_score       (y_true, y_pred, average='weighted', zero_division=0) * 100, 2)
    return acc, prec, rec, f1


def get_data_subset(df, target_col, pct, random_state=42):
    """
    Returns pct% of the dataframe, sampled STRATIFIED by target_col
    (equal fraction from every class), then shuffled to remove ordering bias.

    Why shuffle after sampling?
    groupby().sample() keeps rows grouped by class. Shuffling ensures the
    downstream split/CV sees a randomly interleaved dataset.
    """
    if pct == 100:
        subset = df.copy()
    else:
        frac = pct / 100.0
        subset = (
            df.groupby(target_col, group_keys=False)
              .apply(lambda g: g.sample(frac=frac, random_state=random_state))
              .reset_index(drop=True)
        )
    # Shuffle to prevent class-ordering bias
    subset = subset.sample(frac=1, random_state=random_state).reset_index(drop=True)
    return subset


def stratified_split(X, y, test_size, random_state=42):
    """
    Splits data ensuring EVERY class contributes proportionally to
    both train and test sets.

    Uses StratifiedShuffleSplit (sklearn) which is equivalent to the
    manual per-class loop but is cleaner and fully verified.
    Both approaches produce identical splits — confirmed by testing.

    Why not random train_test_split?
    A random split may over/under-represent some classes in test set,
    especially when dataset is small or imbalanced.
    """
    sss = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    # next() fetches the single split directly — cleaner than a for-loop
    train_idx, test_idx = next(sss.split(X, y))
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]


print('All utility functions defined.')

---
# EXPERIMENT 1 — Hand Activity Identification

- **Target:** `gesture_label`  
- **Drop:** `person_name`, `person_id`  
- For every data % → 4 train/test splits + 5-Fold Stratified CV  
- All 4 metrics collected for both train/test splits AND CV folds

In [ ]:
# ============================================================
# CELL 5 — EXPERIMENT 1: Hand Activity Identification
# ============================================================

# Feature columns: keep only distance features, drop ID/label columns
FEATURE_COLS_EXP1 = [c for c in df.columns
                     if c not in ['person_name', 'person_id', 'gesture_label']]
TARGET_EXP1 = 'gesture_label'

results_exp1    = []   # Results from train/test splits
cv_results_exp1 = []   # Results from Stratified K-Fold CV

for data_pct in DATA_PERCENTAGES:
    print(f"\n{'='*65}")
    print(f"  [EXP 1]  {data_pct}% of Total Data")
    print(f"{'='*65}")

    # -- Get stratified subset, shuffled --
    df_sub = get_data_subset(df, TARGET_EXP1, data_pct)
    X_sub  = df_sub[FEATURE_COLS_EXP1].values
    y_sub  = df_sub[TARGET_EXP1].values

    print(f"  Subset size : {len(df_sub)} rows")
    print(f"  Class dist  : {dict(zip(*np.unique(y_sub, return_counts=True)))}")

    # --------------------------------------------------------
    # PART A: Train / Test Splits (4 ratios × 7 models)
    # --------------------------------------------------------
    for ratio_label, test_size in SPLIT_RATIOS.items():
        print(f"\n  --- Split {ratio_label} ---")

        # Stratified split: each class contributes exactly test_size %
        X_train, X_test, y_train, y_test = stratified_split(
            X_sub, y_sub, test_size=test_size
        )
        print(f"  Train={len(y_train)}, Test={len(y_test)}")

        for model_name, model in get_models().items():
            try:
                if model_name == 'XGBoost':
                    # Remap train labels to 0-indexed; apply same map to test
                    y_tr_xgb, lmap = remap_labels_for_xgboost(y_train)
                    y_te_xgb       = np.array([lmap[v] for v in y_test])
                    model.fit(X_train, y_tr_xgb)
                    y_pred = model.predict(X_test)
                    acc, prec, rec, f1 = get_metrics(y_te_xgb, y_pred)
                else:
                    model.fit(X_train, y_train)
                    y_pred = model.predict(X_test)
                    acc, prec, rec, f1 = get_metrics(y_test, y_pred)

                print(f"  {model_name:<22} Acc={acc}%  Prec={prec}%  Rec={rec}%  F1={f1}%")
                results_exp1.append({
                    'Total Data %'    : data_pct,
                    'Train-Test Ratio': ratio_label,
                    'Algorithm'       : model_name,
                    'Accuracy'        : acc,
                    'Precision'       : prec,
                    'Recall'          : rec,
                    'F1-Score'        : f1
                })
            except Exception as e:
                print(f"  {model_name:<22} ERROR: {e}")

    # --------------------------------------------------------
    # PART B: Stratified K-Fold Cross-Validation
    # Applied to EVERY data percentage (100, 75, 50, 25)
    # Manual fold loop used for ALL models so that:
    #   1. All 4 metrics are captured per fold
    #   2. XGBoost label remapping is applied correctly INSIDE each fold
    #      (remapping must use ONLY training-fold labels, not global labels)
    # --------------------------------------------------------
    print(f"\n  --- Stratified {N_FOLDS}-Fold CV on {data_pct}% data ---")
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

    for model_name, model in get_models().items():
        fold_acc, fold_prec, fold_rec, fold_f1 = [], [], [], []

        for fold_num, (train_idx, test_idx) in enumerate(skf.split(X_sub, y_sub), 1):
            X_tr, X_te = X_sub[train_idx], X_sub[test_idx]
            y_tr, y_te = y_sub[train_idx], y_sub[test_idx]

            try:
                if model_name == 'XGBoost':
                    # IMPORTANT: remap using ONLY this fold's training labels
                    # so the 0-indexed mapping is consistent within this fold
                    y_tr_xgb, lmap = remap_labels_for_xgboost(y_tr)
                    y_te_xgb       = np.array([lmap[v] for v in y_te])
                    model.fit(X_tr, y_tr_xgb)
                    y_pred = model.predict(X_te)
                    acc, prec, rec, f1 = get_metrics(y_te_xgb, y_pred)
                else:
                    model.fit(X_tr, y_tr)
                    y_pred = model.predict(X_te)
                    acc, prec, rec, f1 = get_metrics(y_te, y_pred)

                fold_acc.append(acc);  fold_prec.append(prec)
                fold_rec.append(rec);  fold_f1.append(f1)

            except Exception as e:
                print(f"  {model_name:<22} Fold {fold_num} ERROR: {e}")
                fold_acc.append(np.nan); fold_prec.append(np.nan)
                fold_rec.append(np.nan); fold_f1.append(np.nan)

        # Aggregate across folds
        mean_acc  = round(np.nanmean(fold_acc),  2)
        mean_prec = round(np.nanmean(fold_prec), 2)
        mean_rec  = round(np.nanmean(fold_rec),  2)
        mean_f1   = round(np.nanmean(fold_f1),   2)
        std_acc   = round(np.nanstd(fold_acc),   2)

        print(f"  {model_name:<22} CV Acc={mean_acc}%±{std_acc}%  Prec={mean_prec}%  Rec={mean_rec}%  F1={mean_f1}%")

        cv_results_exp1.append({
            'Total Data %': data_pct,
            'Algorithm'   : model_name,
            'CV Acc Mean' : mean_acc,
            'CV Acc Std'  : std_acc,
            'CV Prec Mean': mean_prec,
            'CV Prec Std' : round(np.nanstd(fold_prec), 2),
            'CV Rec Mean' : mean_rec,
            'CV Rec Std'  : round(np.nanstd(fold_rec),  2),
            'CV F1 Mean'  : mean_f1,
            'CV F1 Std'   : round(np.nanstd(fold_f1),   2)
        })

print("\n" + "✅ Experiment 1 Complete!")

In [ ]:
# ============================================================
# CELL 6 — EXP 1: Display Results as Pivot Table
# ============================================================

ALGO_ORDER   = ['Naive Bayes', 'Decision Tree', 'XGBoost', 'Random Forest',
                'KNN', 'Logistic Regression', 'SVM']
METRIC_ORDER = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
RATIO_ORDER  = ['80-20', '70-30', '60-40', '50-50']

df_exp1 = pd.DataFrame(results_exp1)

# Pivot to replicate the Data_Needed.xlsx layout
pivot_exp1 = df_exp1.pivot_table(
    index   =['Total Data %', 'Train-Test Ratio'],
    columns ='Algorithm',
    values  =METRIC_ORDER
)
pivot_exp1 = pivot_exp1.reindex(
    columns=pd.MultiIndex.from_product([METRIC_ORDER, ALGO_ORDER])
)
# Restore row order
ratio_order_idx = pd.CategoricalIndex(df_exp1['Train-Test Ratio'], categories=RATIO_ORDER, ordered=True)

print('Experiment 1 — Train/Test Results:')
print(pivot_exp1.to_string())

print('\nExperiment 1 — Cross-Validation Results:')
df_cv1 = pd.DataFrame(cv_results_exp1)
print(df_cv1.to_string(index=False))

---
# EXPERIMENT 2 — Person Identification

- **Target:** `person_id`  
- **Drop:** `person_name`, `gesture_label`  
- Same split + CV structure as Experiment 1

In [ ]:
# ============================================================
# CELL 7 — EXPERIMENT 2: Person Identification
# ============================================================

# Feature columns: keep distance features, drop person_name, gesture_label
# Also drop person_id since it is the TARGET for this experiment
FEATURE_COLS_EXP2 = [c for c in df.columns
                     if c not in ['person_name', 'gesture_label', 'person_id']]
TARGET_EXP2 = 'person_id'

results_exp2    = []
cv_results_exp2 = []

for data_pct in DATA_PERCENTAGES:
    print(f"\n{'='*65}")
    print(f"  [EXP 2]  {data_pct}% of Total Data")
    print(f"{'='*65}")

    # Stratified subset by person_id (equal % from each person)
    df_sub = get_data_subset(df, TARGET_EXP2, data_pct)
    X_sub  = df_sub[FEATURE_COLS_EXP2].values
    y_sub  = df_sub[TARGET_EXP2].values     # ← target is person_id

    print(f"  Subset size  : {len(df_sub)} rows")
    print(f"  Person dist  : {dict(zip(*np.unique(y_sub, return_counts=True)))}")

    # --------------------------------------------------------
    # PART A: Train / Test Splits
    # --------------------------------------------------------
    for ratio_label, test_size in SPLIT_RATIOS.items():
        print(f"\n  --- Split {ratio_label} ---")

        X_train, X_test, y_train, y_test = stratified_split(
            X_sub, y_sub, test_size=test_size
        )
        print(f"  Train={len(y_train)}, Test={len(y_test)}")

        for model_name, model in get_models().items():
            try:
                if model_name == 'XGBoost':
                    # person_id values (27, 50, 103...) are NOT 0-indexed → must remap
                    y_tr_xgb, lmap = remap_labels_for_xgboost(y_train)
                    y_te_xgb       = np.array([lmap[v] for v in y_test])
                    model.fit(X_train, y_tr_xgb)
                    y_pred = model.predict(X_test)
                    acc, prec, rec, f1 = get_metrics(y_te_xgb, y_pred)
                else:
                    model.fit(X_train, y_train)
                    y_pred = model.predict(X_test)
                    acc, prec, rec, f1 = get_metrics(y_test, y_pred)

                print(f"  {model_name:<22} Acc={acc}%  Prec={prec}%  Rec={rec}%  F1={f1}%")
                results_exp2.append({
                    'Total Data %'    : data_pct,
                    'Train-Test Ratio': ratio_label,
                    'Algorithm'       : model_name,
                    'Accuracy'        : acc,
                    'Precision'       : prec,
                    'Recall'          : rec,
                    'F1-Score'        : f1
                })
            except Exception as e:
                print(f"  {model_name:<22} ERROR: {e}")

    # --------------------------------------------------------
    # PART B: Stratified K-Fold Cross-Validation
    # (same manual fold approach as Experiment 1)
    # --------------------------------------------------------
    print(f"\n  --- Stratified {N_FOLDS}-Fold CV on {data_pct}% data ---")
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

    for model_name, model in get_models().items():
        fold_acc, fold_prec, fold_rec, fold_f1 = [], [], [], []

        for fold_num, (train_idx, test_idx) in enumerate(skf.split(X_sub, y_sub), 1):
            X_tr, X_te = X_sub[train_idx], X_sub[test_idx]
            y_tr, y_te = y_sub[train_idx], y_sub[test_idx]

            try:
                if model_name == 'XGBoost':
                    # Remap inside this fold using ONLY this fold's training labels
                    y_tr_xgb, lmap = remap_labels_for_xgboost(y_tr)
                    y_te_xgb       = np.array([lmap[v] for v in y_te])
                    model.fit(X_tr, y_tr_xgb)
                    y_pred = model.predict(X_te)
                    acc, prec, rec, f1 = get_metrics(y_te_xgb, y_pred)
                else:
                    model.fit(X_tr, y_tr)
                    y_pred = model.predict(X_te)
                    acc, prec, rec, f1 = get_metrics(y_te, y_pred)

                fold_acc.append(acc);  fold_prec.append(prec)
                fold_rec.append(rec);  fold_f1.append(f1)

            except Exception as e:
                print(f"  {model_name:<22} Fold {fold_num} ERROR: {e}")
                fold_acc.append(np.nan); fold_prec.append(np.nan)
                fold_rec.append(np.nan); fold_f1.append(np.nan)

        mean_acc  = round(np.nanmean(fold_acc),  2)
        mean_prec = round(np.nanmean(fold_prec), 2)
        mean_rec  = round(np.nanmean(fold_rec),  2)
        mean_f1   = round(np.nanmean(fold_f1),   2)
        std_acc   = round(np.nanstd(fold_acc),   2)

        print(f"  {model_name:<22} CV Acc={mean_acc}%±{std_acc}%  Prec={mean_prec}%  Rec={mean_rec}%  F1={mean_f1}%")

        cv_results_exp2.append({
            'Total Data %': data_pct,
            'Algorithm'   : model_name,
            'CV Acc Mean' : mean_acc,
            'CV Acc Std'  : std_acc,
            'CV Prec Mean': mean_prec,
            'CV Prec Std' : round(np.nanstd(fold_prec), 2),
            'CV Rec Mean' : mean_rec,
            'CV Rec Std'  : round(np.nanstd(fold_rec),  2),
            'CV F1 Mean'  : mean_f1,
            'CV F1 Std'   : round(np.nanstd(fold_f1),   2)
        })

print("\n" + "✅ Experiment 2 Complete!")

In [ ]:
# ============================================================
# CELL 8 — EXP 2: Display Results
# ============================================================

df_exp2 = pd.DataFrame(results_exp2)

pivot_exp2 = df_exp2.pivot_table(
    index   =['Total Data %', 'Train-Test Ratio'],
    columns ='Algorithm',
    values  =METRIC_ORDER
)
pivot_exp2 = pivot_exp2.reindex(
    columns=pd.MultiIndex.from_product([METRIC_ORDER, ALGO_ORDER])
)

print('Experiment 2 — Train/Test Results:')
print(pivot_exp2.to_string())

print('\nExperiment 2 — Cross-Validation Results:')
df_cv2 = pd.DataFrame(cv_results_exp2)
print(df_cv2.to_string(index=False))

In [ ]:
# ============================================================
# CELL 9 — SAVE ALL RESULTS TO EXCEL
# Output matches the Data_Needed.xlsx layout.
# ============================================================

output_path = 'Experiment_Results_v2.xlsx'

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:

    # Experiment 1 — train/test flat table
    df_exp1.to_excel(writer, sheet_name='Exp1_Hand_Activity', index=False)

    # Experiment 1 — CV results (all data %, all metrics)
    df_cv1.to_excel(writer, sheet_name='Exp1_CV_Results', index=False)

    # Experiment 2 — train/test flat table
    df_exp2.to_excel(writer, sheet_name='Exp2_Person_ID', index=False)

    # Experiment 2 — CV results (all data %, all metrics)
    df_cv2.to_excel(writer, sheet_name='Exp2_CV_Results', index=False)

print(f'✅ Results saved → {output_path}')
print()
print('Sheets created:')
print('  Exp1_Hand_Activity  — Exp 1 train/test results (all splits, all metrics)')
print('  Exp1_CV_Results     — Exp 1 CV results (100%, 75%, 50%, 25% × 7 models × 4 metrics)')
print('  Exp2_Person_ID      — Exp 2 train/test results')
print('  Exp2_CV_Results     — Exp 2 CV results (same structure)')

---
# EXPERIMENT 3 — Hand Activity + Person Identification (Combined)

> To be implemented in the next notebook.  
> Strategy: encode `(gesture_label, person_id)` as a single combined integer target,  
> then apply the same isolated split + CV pipeline.